In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("jan to may police violation_anonymized791b166.csv")

print(df.shape)
print(df.head())

(298450, 24)
           id   latitude  longitude  \
0  FKID000000  12.925557  77.618665   
1  FKID000001  12.905463  77.700778   
2  FKID000002  12.925449  77.618504   
3  FKID000003  12.956521  77.518618   
4  FKID000004  12.977767  77.580545   

                                            location vehicle_number  \
0  18th Main Road, Block 2, Koramangala, Bengalur...    FKN00GL0000   
1  Sarjapura Main Road, The Grove, Janatha Colony...    FKN00GL0001   
2  Koramangala 2nd Block, Kormangala West, Bengal...    FKN00GL0002   
3  6th Cross Road, Manasa Layout, Nagarbhavi, Ben...    FKN00GL0003   
4  Kalidasa Road, Gandhinagar, Nehru Nagar, Benga...    FKN00GL0004   

  vehicle_type  description                                  violation_type  \
0          CAR          NaN  ["WRONG PARKING","PARKING NEAR ROAD CROSSING"]   
1          CAR          NaN                                  ["NO PARKING"]   
2          CAR          NaN      ["WRONG PARKING","PARKING IN A MAIN ROAD"]   
3      SC

In [2]:
df['created_datetime'] = pd.to_datetime(df['created_datetime'], format='ISO8601')

df['hour'] = df['created_datetime'].dt.hour
df['day'] = df['created_datetime'].dt.day
df['month'] = df['created_datetime'].dt.month
df['weekday'] = df['created_datetime'].dt.day_name()

In [3]:
location_counts = (
    df.groupby('location')
      .size()
      .reset_index(name='violation_count')
)

df = df.merge(location_counts,on='location')

In [4]:
df['risk_level'] = pd.qcut(
    df['violation_count'],
    q=3,
    labels=['Low','Medium','High']
)

In [5]:
from sklearn.preprocessing import LabelEncoder

cat_cols = [
    'vehicle_type',
    'location',
    'junction_name',
    'police_station'
]

encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

In [6]:
from sklearn.model_selection import train_test_split

X = df[
[
'vehicle_type',
'location',
'junction_name',
'police_station',
'hour',
'month'
]
]

y = df['risk_level']

X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [7]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    random_state=42
)

# Convert categorical labels to numerical codes
model.fit(X_train, y_train.cat.codes)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=8,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_parallel_tree=None, ...)

In [8]:
from sklearn.metrics import classification_report

pred = model.predict(X_test)

# Convert y_test to numerical codes to match pred
print(classification_report(y_test.cat.codes,pred))

              precision    recall  f1-score   support

           0       0.94      0.87      0.90     19621
           1       0.89      0.91      0.90     19805
           2       0.95      0.99      0.97     19656

    accuracy                           0.92     59082
   macro avg       0.92      0.92      0.92     59082
weighted avg       0.92      0.92      0.92     59082



In [5]:
import pandas as pd
from sklearn.cluster import MiniBatchKMeans

#from sklearn.cluster import KMeans
df = pd.read_csv("jan to may police violation_anonymized791b166.csv")


coords = df[['latitude','longitude']]

kmeans = MiniBatchKMeans(
    n_clusters=30,
    batch_size=10000,
    random_state=42
)

df['cluster'] = kmeans.fit_predict(coords)

In [2]:
import folium

center = [
    df.latitude.mean(),
    df.longitude.mean()
]

m = folium.Map(
    location=center,
    zoom_start=11
)

for _,row in df.iterrows():

    folium.CircleMarker(
        location=[
            row.latitude,
            row.longitude
        ],
        radius=2,
        color='red'
    ).add_to(m)

m.save("hotspots.html")

In [6]:
import folium

center = [df.latitude.mean(), df.longitude.mean()]

m = folium.Map(location=center, zoom_start=11)

centroids = pd.DataFrame(
    kmeans.cluster_centers_,
    columns=['latitude','longitude']
)

for _, row in centroids.iterrows():

    folium.Marker(
        location=[row.latitude, row.longitude],
        popup='Parking Hotspot',
        icon=folium.Icon(color='red')
    ).add_to(m)

m.save("hotspots2.html")

In [2]:
import pandas as pd
df = pd.read_csv("jan to may police violation_anonymized791b166.csv")
top_locations = (
    df.groupby('location')
      .size()
      .sort_values(ascending=False)
      .head(20)
)

print(top_locations)

location
Unnamed Road, Begur Chikkanahalli, Bengaluru, Karnataka. Pin-562149 (India)                                                4090
Kamaraj Road, Sri Nagamma Devi Circle, Sivanchetti Gardens, Bengaluru, Karnataka. Pin-560042 (India)                       3999
New Horizon College Road, New Horizon College of Engineering, Kadubisanahalli, Bengaluru, Karnataka. Pin-560103 (India)    3785
MBT Road, Devasandra Junction, KR Puram, Bengaluru, Karnataka. Pin-560036 (India)                                          3027
Dispensary Road, Tasker Town, Shivaji Nagar, Bengaluru, Karnataka. Pin-560001 (India)                                      2670
Bellary Road, Vinayaka Nagar, Hebbal, Bengaluru, Karnataka. Pin-560024 (India)                                             2639
5th Main Road, Kempe Gowda Circle, Gandhi Nagar, Bengaluru, Karnataka. Pin-560009 (India)                                  2604
Main Guard Cross Road, Tasker Town, Shivaji Nagar, Bengaluru, Karnataka. Pin-560001 (India)    

In [ ]:
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt

if not hasattr(model, "booster_"):
    import matplotlib.pyplot as plt

    if not hasattr(model, "booster_"):
        if 'X_train' not in globals() or 'y_train' not in globals():
            X = df[
                [
                    'vehicle_type',
                    'location',
                    'junction_name',
                    'police_station',
                    'hour',
                    'month'
                ]
            ]
            y = df['risk_level']
            X_train, X_test, y_train, y_test = train_test_split(
                X,
                y,
                test_size=0.2,
                random_state=42
            )
        model.fit(X_train, y_train.cat.codes)

    importance = model.feature_importances_

    features = X.columns

    plt.figure(figsize=(8,5))
    plt.barh(features, importance)
    plt.xlabel("Importance")
    plt.title("Feature Importance")
    plt.show()

importance = model.feature_importances_

features = X.columns

plt.figure(figsize=(8,5))
plt.barh(features, importance)
plt.xlabel("Importance")
plt.title("Feature Importance")
plt.show()

NameError: name 'X_train' is not defined